In [ ]:
#| default_exp window

In [ ]:
#| hide
import socket
from http.server import BaseHTTPRequestHandler, HTTPServer
from importlib.util import find_spec
from threading import Event, Thread
from fastcore.test import *
from nbdev.showdoc import *

This module is the part of kavacha that runs inside the finished app. It opens a native window that shows a web app served from a loopback address, one on the same machine such as 127.0.0.1, and gives that window what people expect of a desktop app: a menu bar, a Dock menu of recent folders, a window that hides instead of quitting when it is closed, and a response when a folder is dropped on the app's icon. pywebview draws the window, and most of the platform-specific parts are for macOS.

The application that uses kavacha is called the host here. It supplies its name, its menus and the callbacks that do its own work, such as opening a folder. `run_shell`, near the end, is the function a host calls. Everything above it is a piece `run_shell` is built from.

In [ ]:
#| export
from __future__ import annotations

In [ ]:
#| export
import os, re, sys, threading, time

In [ ]:
#| export
from fastcore.all import Path

In [ ]:
#| export
from fastcore.xdg import xdg_config_home

In [ ]:
#| export
from kavacha.menus import PC_NATIVE, RESERVED, Binding, Js, MenuItem, Std, _chord, chord_glyph, gtk_accel, pc_chord, pc_key, pc_reserved

In [ ]:
#| export
APP_NAME = os.environ.get('KAVACHA_APP_NAME') or 'app'
ENV_PREFIX = 'KAVACHA_'

def use_app(name, cfg=None, env_prefix='KAVACHA_'):
    "Set the app name, the config and storage folders, and the environment variable prefix kavacha uses."
    global APP_NAME, CFG, STORAGE, ENV_PREFIX, EAGER_CDP
    APP_NAME, ENV_PREFIX = str(name or 'app'), str(env_prefix or '')
    CFG = Path(cfg) if cfg else Path(os.environ.get(f'{ENV_PREFIX}CFG') or xdg_config_home()/APP_NAME)
    STORAGE, EAGER_CDP = CFG/'webview', f'{ENV_PREFIX}CDP_EAGER'
    return APP_NAME

In [ ]:
#| export
CFG = Path(os.environ.get('KAVACHA_CFG') or xdg_config_home()/APP_NAME)

A few things are named after the app: the folder where the window keeps its settings and browser storage, the title on the holding page, and the prefix of the environment variables kavacha reads. `APP_NAME` is that name. It is read from `$KAVACHA_APP_NAME` when the module is imported and defaults to `app`. `CFG` is the settings folder, from `$KAVACHA_CFG` or else the XDG config folder named after the app, such as `~/.config/app`.

A host should call `use_app` once, before it opens a window, with its own name. It sets `APP_NAME`, `CFG`, `STORAGE` and the environment variable prefix together. After `use_app('leela', env_prefix='LEELA_')`, for example, the window size comes from `$LEELA_WINDOW_SIZE` instead of `$KAVACHA_WINDOW_SIZE`, and the settings folder from `$LEELA_CFG`. None of this creates anything on disk.

In [ ]:
#| export
def errstr(e): return f"{type(e).__name__}: {e}"

`errstr` formats an exception as its type and its message. Every warning this module prints uses it. The type is kept because many messages say little on their own: a `KeyError` carries only the missing key.

In [ ]:
#| exec_doc
errstr(ValueError('port 9223 is in use')), errstr(KeyError('cocoa'))

In [ ]:
#| export
BACKENDS = {'darwin': 'cocoa', 'win32': 'edgechromium', 'linux': 'gtk'}

In [ ]:
#| export
DEFAULT_SIZE = (1440, 900)

In [ ]:
#| export
MIN_SIZE = (900, 560)

In [ ]:
#| export
STORAGE = CFG/'webview'

In [ ]:
#| export
EAGER_CDP = 'KAVACHA_CDP_EAGER'

`DEFAULT_SIZE` is a new window's size when nothing else is asked for. `MIN_SIZE` is the smallest size the user can drag a window to, and the smallest `window_size` accepts. `STORAGE` is the folder inside `CFG` where the web view keeps cookies and local storage between runs. `EAGER_CDP` holds the name of the environment variable `warm_cdp` reads, described below. `BACKENDS` is the same table as in `kavacha.probe`.

In [ ]:
#| exec_doc
DEFAULT_SIZE, MIN_SIZE, str(STORAGE).replace(str(CFG), '/cfg')

In [ ]:
#| export
def is_colour(c):
    "Whether `c` is a `#rrggbb` colour."
    return isinstance(c, str) and re.fullmatch(r'#[0-9a-fA-F]{6}', c) is not None

def paint_background(window, colour):
    "Give pywebview `window` the native background `colour`. On macOS an open window repaints now."
    window.background_color = colour
    if (hwnd := getattr(window, 'hwnd', None)) is not None: tint_caption(hwnd, colour)
    if sys.platform != 'darwin': return
    from PyObjCTools.AppHelper import callAfter
    from webview.platforms.cocoa import BrowserView
    def go():
        if (bv := BrowserView.instances.get(window.uid)) is None: return
        c = BrowserView.nscolor_from_hex(colour)
        bv.window.setBackgroundColor_(c)
        # What WebKit draws past the page's edge while it catches up with a resize.
        if bv.webview.respondsToSelector_('setUnderPageBackgroundColor:'): bv.webview.setUnderPageBackgroundColor_(c)
    callAfter(go)

In [ ]:
#| export
class ShellApi:
    "What the page may ask the native window for, as `window.pywebview.api.*`."
    on_open_folder = on_open_file = on_page_folder = on_picked = None
    # pywebview gives the page every function `dir` lists, and a host's callbacks skip the questions the page must get.
    def __dir__(self): return [n for n in super().__dir__() if not n.startswith('on_')]
    def open_folder(self, path):
        "Ask the host to open `path` in a window of its own, and return whether a host callback took it."
        if not path or (f := self.on_page_folder or self.on_open_folder) is None: return False
        f(str(path))
        return True
    def open_path(self, path):
        "Open a folder as `open_folder` does, and a file through `on_open_file` as Finder would. Whether a callback took it."
        if not path: return False
        if os.path.isdir(path): return self.open_folder(path)
        if self.on_open_file is None: return False
        self.on_open_file(str(path))
        return True
    def _pick(self, choose, what, folder=False):
        import webview
        if (window := webview.active_window()) is None: return None
        try: picked = choose(window)
        except Exception as e:
            print(f'  {what} dialog: {errstr(e)}')
            return None
        if not picked: return None
        p = str(picked[0])
        if self.on_picked and (folder or os.path.isdir(p)): self.on_picked(p)
        return p
    def pick_folder(self):
        "The platform's own folder chooser. A path, or None if it was cancelled."
        import webview
        return self._pick(lambda w: w.create_file_dialog(webview.FileDialog.FOLDER), 'folder', True)
    def pick_path(self):
        "The platform's own chooser for a file or a folder. A path, or None if it was cancelled."
        return self._pick(choose_path, 'file')
    def dropped(self, names, take=True):
        "Files dropped on the page from outside: with `take`, open them as Finder would. The paths it opened."
        paths = take_dropped(names)
        if take and paths: open_paths(paths, self.on_open_folder, self.on_open_file)
        return paths if take else []
    on_recent = None
    def recent(self):
        "Whatever the host offers as recent folders, or nothing."
        return list(self.on_recent()) if self.on_recent else []
    unified = False
    #: False draws no title text in a native macOS title bar; the title is still set, so the Window menu and accessibility read it
    title_text = True
    def titlebar(self):
        "How this app's windows show their title bar, as `titlebar_mode` names it: `native` unless the host set `unified`."
        return titlebar_mode() if self.unified else 'native'
    background = None
    def set_background(self, colour):
        "Paint every open window's native background `colour`, and open later windows on it. False unless `#rrggbb`."
        if not is_colour(colour): return False
        self.background = colour
        try: import webview
        except ImportError: return True
        for w in list(webview.windows): paint_background(w, colour)
        return True


The page in the window sometimes needs the operating system, for example to show the native folder chooser. pywebview allows this through a JavaScript API: the methods of a Python object become functions the page calls as `window.pywebview.api.<method>()`. `ShellApi` is that object.

`pick_folder` shows the platform's folder chooser and returns the chosen path, or `None` if the user cancels. `pick_path` does the same with a chooser that takes a file or a folder, as File > Open does in a Mac app. `open_folder` and `recent` pass the request to the host's `on_open_folder` and `on_recent` callbacks, which `run_shell` sets. Without a callback, `open_folder` returns `False` and `recent` returns an empty list. `open_path` takes either kind: a folder goes the way of `open_folder`, and a file goes to `on_open_file`, the callback a file opened from Finder reaches. A host that must tell a request from the page apart from a Finder or Dock open sets `on_page_folder`: `open_folder` then calls it in place of `on_open_folder`. `on_picked` gets each folder the person chooses in `pick_folder` or `pick_path`, before the page sees it, so the host can trust that one path. It is not called for a file or when the person cancels. pywebview calls these methods on its own worker threads, so the callbacks must be safe to call from any thread. `dropped` is for files dropped on the window, described with `arm_drops` below. The page cannot call the `on_` callbacks themselves: pywebview offers the page every function an object lists, so `ShellApi` lists none of them.

`set_background` is for a page with themes. It takes the page's background as `#rrggbb` and paints every open window's native background with it, which is what shows while the window opens or grows faster than the page redraws. It also keeps the colour in `background`, so a window opened later starts on it. The colour is per app, not per window: pywebview does not say which window called. Anything that is not `#rrggbb` returns `False` and changes nothing.


In [ ]:
#| exec_doc
api = ShellApi()
opened = []
api.on_open_folder = opened.append
api.open_folder('/proj/demo'), api.open_folder(''), opened

A page request goes to `on_page_folder` when the host sets it. Finder and the Dock still use `on_open_folder`.

In [ ]:
two, page = ShellApi(), []
two.on_open_folder, two.on_page_folder = opened.append, page.append
two.open_folder('/proj/two')
test_eq(page, ['/proj/two'])
test_eq(opened, ['/proj/demo'])

`on_picked` gets the folder the person chose, and is not called on a cancel. A fake `webview` module stands in for the real chooser.

In [ ]:
import sys, tempfile, types
from unittest.mock import patch

In [ ]:
def fake_webview(answer): return types.SimpleNamespace(FileDialog=types.SimpleNamespace(FOLDER=20), active_window=lambda: types.SimpleNamespace(create_file_dialog=lambda kind: answer))
picked = []
two.on_picked = picked.append
with patch.dict(sys.modules, webview=fake_webview(['/proj/chosen'])): test_eq(two.pick_folder(), '/proj/chosen')
with patch.dict(sys.modules, webview=fake_webview(None)): test_eq(two.pick_folder(), None)
test_eq(picked, ['/proj/chosen'])

In [ ]:
#| hide
test_eq(ShellApi().open_folder('/proj/demo'), False)   
test_eq(ShellApi().recent(), [])
api.on_recent = lambda: iter(['/proj/demo', '/proj/notes'])
test_eq(api.recent(), ['/proj/demo', '/proj/notes'])   
api.open_folder(Path('/proj/notes'))
test_eq(opened[-1], '/proj/notes')                     
test_eq([ShellApi().set_background(c) for c in ('red', '#12345', None, 'url(x)')], [False]*4)
api.set_background('#221d18')
test_eq(api.background, '#221d18')

In [ ]:
#| export
def choose_path(window):
    "Paths from a chooser that takes one file or one folder: `NSOpenPanel` on macOS, pywebview's file dialog elsewhere."
    import webview
    if sys.platform != 'darwin': return window.create_file_dialog(webview.FileDialog.OPEN)
    import AppKit
    from PyObjCTools.AppHelper import callAfter
    got, done = [], threading.Event()
    def go():
        try:
            p = AppKit.NSOpenPanel.openPanel()
            p.setCanChooseFiles_(True); p.setCanChooseDirectories_(True); p.setAllowsMultipleSelection_(False)
            if p.runModal() == AppKit.NSModalResponseOK: got.extend(str(u.path()) for u in p.URLs())
        finally: done.set()
    if threading.current_thread() is threading.main_thread(): go()
    else: callAfter(go); done.wait()
    return got


pywebview's own dialog chooses files or folders, never both, so `choose_path` builds the macOS panel itself, on the main thread where AppKit wants it, and waits for the answer. Off macOS it shows the file dialog, which chooses files only. `pick_path` calls `on_picked` only when the choice is a folder, and `open_path` sends it on by its kind. A stand-in `choose_path` shows both.

In [ ]:
_d2 = tempfile.mkdtemp(); _f2 = os.path.join(_d2, 'a.png'); open(_f2, 'w').close()
three, files, folders, picked = ShellApi(), [], [], []
three.on_open_file, three.on_page_folder, three.on_picked = files.append, folders.append, picked.append
for got in ([_f2], [_d2], [], None):
    with patch.dict(sys.modules, webview=fake_webview(None)), patch(f'{__name__}.choose_path', lambda w, got=got: got):
        p = three.pick_path()
        if p: three.open_path(p)
test_eq((files, folders, picked), ([_f2], [_d2], [_d2]))
test_eq((three.open_path(''), ShellApi().open_path(_f2)), (False, False))
test_eq([n for n in dir(three) if n.startswith('on_')], [])
assert {'open_path', 'pick_path', 'dropped'} <= set(dir(three))


In [ ]:
#| export
def backend(platform=None):
    "pywebview's GUI name for `platform`, or None where there is no supported webview."
    return BACKENDS.get(platform or sys.platform)

In [ ]:
#| export
def shell_ready(platform=None):
    "`(ok, why)`: whether a native window can be opened here, and what is missing if not."
    gui = backend(platform)
    if gui is None: return False, f'no native webview backend for {platform or sys.platform}'
    try: import webview  # noqa: F401
    except ImportError as e: return False, f'pywebview is not installed ({errstr(e)})'
    return True, gui

`backend` and `shell_ready` make the same checks as their namesakes in `kavacha.probe`: which pywebview GUI this platform uses, and whether a window can be opened here. `run_shell` calls `shell_ready` first and raises its reason as a `RuntimeError` when the answer is no.

In [ ]:
#| exec_doc
backend('darwin'), backend('win32'), backend('plan9')

In [ ]:
#| exec_doc
test_eq(shell_ready('plan9'), (False, 'no native webview backend for plan9'))
ok, why = shell_ready('darwin')
test_eq(ok, why == 'cocoa')

In [ ]:
#| export
def wait_for_http(url, timeout=60, interval=.1):
    "Wait until `url` gives any HTTP response or `timeout` seconds pass, and return whether it did."
    import urllib.request
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            urllib.request.urlopen(url, timeout=.5).close()
            return True
        except urllib.error.HTTPError: return True
        except Exception: time.sleep(interval)
    return False

`wait_for_http` waits until the server at `url` answers or `timeout` seconds pass, and returns whether it answered. Unlike the version in `kavacha.probe`, any HTTP response counts, including an error such as 404, because a server that sends an error page has started. Only a failed connection means it is not ready. The example asks for a page that does not exist and still gets `True`.

In [ ]:
#| hide
#| exec_doc
class _Root(BaseHTTPRequestHandler):
    def do_GET(self): self.send_response(200 if self.path == '/' else 404); self.end_headers()
    def log_message(self, *a): pass
srv = HTTPServer(('127.0.0.1', 0), _Root); Thread(target=srv.serve_forever, daemon=True).start()
url = f'http://127.0.0.1:{srv.server_port}/'
s = socket.socket(); s.bind(('127.0.0.1', 0)); dead = f'http://127.0.0.1:{s.getsockname()[1]}/'; s.close()

In [ ]:
#| exec_doc
wait_for_http(url), wait_for_http(url + 'missing', timeout=.3)

In [ ]:
#| hide
test_eq(wait_for_http(dead, timeout=.3, interval=.05), False) 

In [ ]:
#| export
CDP_PORT = 9223                                                    #: the visible CDP Chrome's port, and fastcdp's
HEADLESS_PORT = 9224                                               #: a headless Chrome's port
HEADLESS_PROFILE = Path.home()/'.cache'/'kavacha'/'headless-chrome'  #: a headless Chrome's own user data folder

def cdp_target(port=None, headless=True, profile=None):
    "The `(port, profile)` a CDP Chrome starts on: a headless one never on the visible Chrome's port or profile."
    if not headless: return port or CDP_PORT, profile
    return (HEADLESS_PORT if port in (None, CDP_PORT) else port), str(profile or HEADLESS_PROFILE)

def start_cdp(port=None, headless=True, wait=False, timeout=None, profile=None):
    "Start a Chrome controlled over CDP on a background thread, through fossick, and return the thread."
    port, profile = cdp_target(port, headless, profile)
    def boot():
        try:
            from fossick.cdp import debug_ready, cdp_setup
            from fossick.core import syncy
            if debug_ready(port): return
            syncy(cdp_setup(port=port, headless=headless, timeout=timeout, user_data_dir=profile))
        except Exception as e: print(f'  inline browser unavailable: {errstr(e)}')
    t = threading.Thread(target=boot, daemon=True, name='kavacha-cdp-boot')
    t.start()
    if wait: t.join(timeout or 30)
    return t

In [ ]:
#| export
def warm_cdp(port=None, profile=None):
    "Start the CDP Chrome at launch, but only when the `EAGER_CDP` environment variable is 1, true or yes."
    if os.environ.get(EAGER_CDP, '').strip() not in ('1', 'true', 'yes'): return None
    return start_cdp(port, headless=True, profile=profile)

Some hosts show web pages inside the app through a separate Chrome that they control over the Chrome DevTools Protocol (CDP), using fossick. `start_cdp` starts that Chrome on `port` in a background thread and returns the thread. If a Chrome is already listening on `port`, it does nothing. If fossick is not installed or Chrome fails to start, it prints a message and the app carries on without it. Pass `wait=True` to wait for the start, for up to `timeout` seconds or 30 by default. `profile` is the Chrome user data folder.

`cdp_target` picks the port and the profile. A visible Chrome uses `port`, or `CDP_PORT` (9223), and `profile`, or fastcdp's own profile. A headless Chrome uses `HEADLESS_PORT` (9224) and `HEADLESS_PROFILE`. It never uses port 9223 or fastcdp's profile, because those belong to the visible CDP Chrome. A headless Chrome on that port and profile takes over the visible Chrome: a click on its app then opens nothing. fossick starts Chrome through fastcdp's `chrome_bin`, so the `FASTCDP_CHROME` environment variable chooses the browser.

Starting Chrome takes time and memory, so a host normally starts it the first time it is needed. `warm_cdp` starts it at launch instead, but only when the environment variable named in `EAGER_CDP`, `KAVACHA_CDP_EAGER` unless `use_app` changed the prefix, is `1`, `true` or `yes` in lower case. Otherwise it returns `None`.

In [ ]:
#| hide
# a headless Chrome never takes the visible CDP Chrome's port or profile; a visible one keeps them
test_eq(cdp_target(), (HEADLESS_PORT, str(HEADLESS_PROFILE)))
test_eq(cdp_target(CDP_PORT), (HEADLESS_PORT, str(HEADLESS_PROFILE)))
test_eq(cdp_target(9300, profile='/tmp/p'), (9300, '/tmp/p'))
test_eq(cdp_target(headless=False), (CDP_PORT, None))
test_eq(cdp_target(9300, headless=False, profile='/tmp/p'), (9300, '/tmp/p'))
if find_spec('fossick') is None:
    for v in ('1', 'true', 'yes', ' yes '):
        os.environ[EAGER_CDP] = v
        t = warm_cdp(); assert t is not None, v
        t.join(10); assert not t.is_alive(), 'the boot thread ends rather than hanging'
for v in ('', '0', 'no', 'TRUE'):
    os.environ[EAGER_CDP] = v
    test_is(warm_cdp(), None)
os.environ.pop(EAGER_CDP, None)
test_is(warm_cdp(), None)

In [ ]:
#| export
def window_size(spec=None):
    "`WIDTHxHEIGHT` from `spec` or `$<prefix>WINDOW_SIZE`, else the default. See `use_app`."
    spec = spec or os.environ.get(f'{ENV_PREFIX}WINDOW_SIZE', '')
    try:
        w, h = (int(n) for n in str(spec).lower().split('x', 1))
        if w >= MIN_SIZE[0] and h >= MIN_SIZE[1]: return w, h
    except (ValueError, TypeError): pass
    return DEFAULT_SIZE

`window_size` decides how big a new window is. It reads `WIDTHxHEIGHT` from `spec`, or from the environment variable `<prefix>WINDOW_SIZE` (`KAVACHA_WINDOW_SIZE` unless `use_app` set another prefix), and accepts an upper or lower case `x`. It returns `DEFAULT_SIZE` when neither is set, when the text cannot be read as a size, or when the size is smaller than `MIN_SIZE`.

In [ ]:
#| exec_doc
window_size('1600x1000'), window_size('1280X800'), window_size('320x200'), window_size('wide')

In [ ]:
#| hide
os.environ['KAVACHA_WINDOW_SIZE'] = '1000x700'
test_eq(window_size(), (1000, 700))
test_eq(window_size('1600x1000'), (1600, 1000)) 
os.environ['KAVACHA_WINDOW_SIZE'] = 'huge'
test_eq(window_size(), DEFAULT_SIZE)
del os.environ['KAVACHA_WINDOW_SIZE']
test_eq(window_size(), DEFAULT_SIZE)
test_eq(window_size('900x560'), MIN_SIZE)       
test_eq(window_size('900x559'), DEFAULT_SIZE)
test_eq(window_size('1600x1000x900'), DEFAULT_SIZE)
test_eq(window_size((1600, 1000)), DEFAULT_SIZE)

In [ ]:
#| export
SPLASH = """<!doctype html><meta charset=utf-8><title>{name}</title>
<style>html,body{{height:100%;margin:0;background:{bg};color:{ink};
font:15px/1.5 -apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;
display:flex;align-items:center;justify-content:center}}
div{{text-align:center;opacity:.85}}b{{display:block;font-size:34px;letter-spacing:-.02em;margin-bottom:10px}}
i{{font-style:normal;font-size:13px;opacity:.6}}</style>
<div><b>{name}</b><i style="{style}">{note}</i></div>"""

def is_light(colour):
    "Whether `#rrggbb` is a light colour, so dark ink reads on it."
    r, g, b = (int(colour[i:i+2], 16) for i in (1, 3, 5))
    return .299*r + .587*g + .114*b > 140

def splash(name=None, note='starting the workspace…', style='', background='#171b20'):
    "Return the holding page a window shows until its server answers, titled with the app's name."
    ink = '#1f2328' if is_light(background) else '#f4f7f9'
    return SPLASH.format(name=name or APP_NAME, note=note, style=style, bg=background, ink=ink)

In [ ]:
#| hide
import kavacha.window as _w
test_eq(_w.APP_NAME, 'app')
assert '<b>app</b>' in _w.splash() and '<title>app</title>' in _w.splash()
try:
    _w.use_app('leela', None, 'LEELA_')
    assert '<b>leela</b>' in _w.splash() and '<title>leela</title>' in _w.splash()
    test_eq(_w.EAGER_CDP, 'LEELA_CDP_EAGER')
    test_eq(_w.STORAGE.parent.name, 'leela')
    assert 'did not answer' in _w._unreachable('http://x')
    assert 'background:#efe6d6;color:#1f2328' in _w.splash(background='#efe6d6')
    assert 'background:#221d18;color:#f4f7f9' in _w.splash(background='#221d18')
    assert '#b3261e' in _w._unreachable('http://x', '#ffffff') and '#e06c75' in _w._unreachable('http://x')
finally: _w.use_app('app')
test_eq((_w.APP_NAME, _w.EAGER_CDP), ('app', 'KAVACHA_CDP_EAGER'))

In [ ]:
#| export
def _unreachable(url, background='#171b20'):
    "The same splash, saying why nothing arrived."
    red = '#b3261e' if is_light(background) else '#e06c75'
    return splash(note=f'the workspace at {url} did not answer', style=f'color:{red}', background=background)

A window opens at once, but its server may take a few seconds to start. Until it does, the window shows a holding page made by `splash`: the app's name in large type and a short note under it, on the window's background, which is dark unless the host gives another. On a light background the text is dark. The page is self-contained and loads nothing from the network. If the server never answers, the window shows the same page with the note replaced, in red, by the address that did not answer. `_unreachable` makes that version, and the example shows its last line.

In [ ]:
#| exec_doc
_unreachable('http://127.0.0.1:8000/').splitlines()[-1]

In [ ]:
#| hide
html = _unreachable('http://127.0.0.1:8000/')
assert 'starting the workspace' not in html, 'the waiting line is replaced, not appended'
assert 'http' not in SPLASH, 'the splash fetches nothing'
test_eq(len(html.splitlines()), len(SPLASH.splitlines()))

In [ ]:
#| export
DOCK_RECENT = 8

In [ ]:
#| export
_dock_target = None

In [ ]:
#| export
def _recent_target(on_folder):
    "One for the life of the app: a menu holds no reference to a Python object, so this must."
    global _dock_target
    if _dock_target is None: _dock_target = RecentTarget.alloc().init_with(on_folder)
    return _dock_target

In [ ]:
#| export
def dock_menu(on_folder, recent=None):
    "The Dock's own menu: the folders last open, newest first. `recent` is the host's list of paths."
    from AppKit import NSMenu, NSMenuItem
    rows = list(recent or ())[:DOCK_RECENT]
    menu = NSMenu.alloc().init()
    if not rows:
        menu.addItem_(_disabled('No recent folders'))
        return menu
    target = _recent_target(on_folder)
    for path in rows:
        path = str(path)
        item = NSMenuItem.alloc().initWithTitle_action_keyEquivalent_(
            os.path.basename(path.rstrip('/')) or path, b'openRecent:', '')
        item.setToolTip_(path)
        item.setRepresentedObject_(path)
        item.setTarget_(target)
        menu.addItem_(item)
    return menu

In [ ]:
#| export
def _disabled(title):
    from AppKit import NSMenuItem
    item = NSMenuItem.alloc().initWithTitle_action_keyEquivalent_(title, None, '')
    item.setEnabled_(False)
    return item

On macOS, right-clicking an app's Dock icon shows a menu. `dock_menu` fills it with the folders the user opened most recently, newest first, up to `DOCK_RECENT` of them. Each row shows the folder's name, with the full path as its tooltip, and choosing one calls `on_folder` with the path. `recent` is the host's list of paths, already in order. With no recent folders the menu holds a single disabled row that says so. The function needs AppKit, so there is no example here.

In [ ]:
#| export
#: Windows closed but kept alive for the Dock icon to bring back, most recent last.
hidden_windows = []
_hidden = hidden_windows
_on_folder, _delegates, _on_recent = None, None, None

In [ ]:
#| export
def dock_delegates():
    "The two delegate subclasses. Built once: an Objective-C class name is global to the runtime."
    global _delegates
    if _delegates is not None: return _delegates
    from AppKit import NSApplicationDelegateReplyFailure, NSApplicationDelegateReplySuccess
    from Foundation import NO, YES
    from webview.platforms.cocoa import BrowserView
    class Windows(BrowserView.WindowDelegate):
        def windowShouldClose_(self, window):
            window.orderOut_(None)
            if window in hidden_windows: hidden_windows.remove(window)
            hidden_windows.append(window)
            return NO
    class App(BrowserView.AppDelegate):
        def applicationShouldHandleReopen_hasVisibleWindows_(self, app, visible):
            if not visible and hidden_windows: hidden_windows.pop().makeKeyAndOrderFront_(None)
            return YES
        def application_openFiles_(self, app, paths):
            ok = open_paths(paths, shell_api().on_open_folder, shell_api().on_open_file)
            app.replyToOpenOrPrint_(NSApplicationDelegateReplySuccess if ok else NSApplicationDelegateReplyFailure)
        def application_openFile_(self, app, path):
            return open_paths([path], shell_api().on_open_folder, shell_api().on_open_file)
        def applicationWillTerminate_(self, note): quit_cleanly()
        def applicationDockMenu_(self, app):
            try: return dock_menu(_on_folder, _on_recent() if _on_recent else None)
            except Exception as e: print(f'  dock menu: {errstr(e)}'); return None
    _delegates = (Windows, App)
    return _delegates
_dock_delegates = dock_delegates

Most macOS apps keep running when their last window is closed, and clicking the Dock icon brings the window back. kavacha does the same. Closing a window hides it, and clicking the Dock icon shows the window that was hidden most recently. Quitting with Cmd+Q still ends the app, after running `quit_cleanly`. This works by replacing pywebview's window and app delegates, the objects AppKit asks how to respond to events such as a window closing; `dock_delegates` builds the two replacement classes once, and `keep_running_in_dock`, below, installs them. The hidden windows wait in `hidden_windows`, the most recently hidden last.

In [ ]:
#| export
#: Set once a quit begins, so a host's background work can tell being reaped from failing.
quitting = threading.Event()

def _reap(grace=1.):
    "Terminate this process's multiprocessing children, give them one shared `grace` seconds, then kill the rest."
    import multiprocessing as mp
    kids = mp.active_children()
    def each(f):
        for p in kids:
            try: f(p)
            except (OSError, ValueError, AssertionError): pass
    each(lambda p: p.terminate())
    end = time.time() + grace
    each(lambda p: p.join(max(0, end - time.time())))
    each(lambda p: p.is_alive() and (p.kill(), p.join(max(0, end + 1 - time.time()))))
    return kids

def _let_sigterm_kill():
    "A second SIGTERM must kill a quit that hangs, so the no-op handler and the wakeup socket go."
    import signal
    if threading.current_thread() is not threading.main_thread(): return
    signal.signal(signal.SIGTERM, signal.SIG_DFL)
    signal.set_wakeup_fd(-1)

def quit_cleanly(timeout=15, run_exit=None, grace=1.):
    "Stop this process's multiprocessing children, then run its atexit handlers, which a Cocoa quit's C `exit()` skips."
    import atexit
    quitting.set()
    n = 0
    for step in (_let_sigterm_kill, lambda: _reap(grace)):
        # Each step on its own: an exception in `applicationWillTerminate_` skips the rest of the quit.
        try: n += len(step() or ())
        except Exception as e: print(f'  quit: {errstr(e)}')
    # On a thread, so a handler that hangs cannot hold the quit past `timeout`.
    t = threading.Thread(target=run_exit or atexit._run_exitfuncs, daemon=True, name='kavacha-quit')
    t.start(); t.join(timeout)
    # Again: a thread still running may have forked a pool meanwhile, and C `exit()` would orphan it.
    try: n += len(_reap(grace))
    except Exception as e: print(f'  quit: {errstr(e)}')
    return n

_sigterm = None

def _die_with(ppid, every=1.):
    "Exit once the parent is gone. A Force Quit or a crash leaves nobody to reap a forked worker."
    while os.getppid() == ppid: time.sleep(every)
    os._exit(1)

def _unhook():
    "In any forked child: SIGTERM kills again and the parent's wakeup socket is let go. Async-safe work only."
    global _sigterm
    import signal
    signal.signal(signal.SIGTERM, signal.SIG_DFL)
    signal.set_wakeup_fd(-1)
    for s in _sigterm or (): s.close()
    _sigterm = None

def _watch_parent(_):
    "In a multiprocessing child only: die with the parent. A deliberately detached fork is left alone."
    import multiprocessing as mp
    if (p := mp.parent_process()) is not None:
        threading.Thread(target=_die_with, args=(p.pid,), daemon=True, name='kavacha-orphan').start()

def quit_on_sigterm():
    "Make SIGTERM quit through Cocoa, so `applicationWillTerminate_` cleans up as Cmd+Q does. macOS, main thread."
    global _sigterm
    import signal, socket
    from multiprocessing.util import register_after_fork
    if sys.platform != 'darwin' or _sigterm or threading.current_thread() is not threading.main_thread(): return _sigterm
    r, w = _sigterm = socket.socketpair()
    w.setblocking(False)
    # The handler never runs while AppKit owns the main thread; the wakeup byte reaches the reader anyway.
    signal.signal(signal.SIGTERM, lambda *_: None)
    signal.set_wakeup_fd(w.fileno(), warn_on_full_buffer=False)
    # Forked children inherit both, and would ignore `terminate()` and wake this reader with their own SIGTERM.
    os.register_at_fork(after_in_child=_unhook)
    register_after_fork(_watch_parent, _watch_parent)
    def wait():
        while signal.SIGTERM not in (got := r.recv(64)):
            if not got: return
        from AppKit import NSApplication
        from PyObjCTools.AppHelper import callAfter
        callAfter(NSApplication.sharedApplication().terminate_, None)
    threading.Thread(target=wait, daemon=True, name='kavacha-sigterm').start()
    return _sigterm

When a Cocoa app quits, it ends the process with C's `exit()`, which skips Python's atexit handlers, and worker processes can outlive it. A worker pool started by forking, as `fastcore.parallel` does on macOS, keeps pipes open that hold its children alive. `quit_cleanly` terminates each child process of this one, kills any still running after one shared `grace` of a second, then runs the atexit handlers on a thread and waits at most `timeout` seconds for them. It reaps once more afterwards, for a pool a still-running thread started meanwhile, and returns the number of children it stopped. It sets `quitting` first, so a host's background job can tell being reaped from failing. `run_exit` replaces the atexit handlers, which is how the test below checks it without exiting.

`quit_on_sigterm` makes a SIGTERM, the signal `kill` sends by default, quit the app the way Cmd+Q does, so that the same clean-up runs. It works on macOS only, must be called on the main thread, and is called by `run_shell`. A forked child gets the default SIGTERM back and lets go of the wakeup socket, and a multiprocessing child exits once its parent is gone, so a Force Quit or a crash leaves no idle worker behind. Once a quit begins, a second SIGTERM kills.

In [ ]:
import multiprocessing as mp, subprocess, tempfile

In [ ]:
kid = mp.get_context('fork').Process(target=time.sleep, args=(60,)); kid.start()
ran = []
test_eq(quit_cleanly(run_exit=lambda: ran.append(1)), 1)
test_eq((kid.is_alive(), ran), (False, [1]))
t0 = time.time(); quit_cleanly(timeout=.3, run_exit=lambda: time.sleep(30))
assert time.time() - t0 < 5, 'a hung handler does not hold the quit'

In [ ]:
_fork_probe = r'''
import os, signal, sys, time, multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from kavacha.window import quit_on_sigterm, quit_cleanly
def ignore(): signal.signal(signal.SIGTERM, signal.SIG_IGN); time.sleep(60)
def nested(path):
    ex = ProcessPoolExecutor(2, mp_context=mp.get_context('fork')); list(ex.map(abs, range(4)))
    open(path, 'w').write(' '.join(str(p.pid) for p in mp.active_children())); time.sleep(60)
def alive(pid):
    try: os.kill(pid, 0); return True
    except OSError: return False
if __name__ == '__main__':
    r, w = quit_on_sigterm(); r.setblocking(False)
    fork = mp.get_context('fork')
    kid = fork.Process(target=time.sleep, args=(60,)); kid.start(); time.sleep(.3)
    kid.terminate(); kid.join(5)
    assert kid.exitcode == -signal.SIGTERM, kid.exitcode
    try: assert not r.recv(64), "a child's SIGTERM reached the parent's wakeup socket"
    except BlockingIOError: pass
    ex = ProcessPoolExecutor(4, mp_context=fork); list(ex.map(abs, range(8)))
    pool = [p.pid for p in mp.active_children()]
    stubborn = [fork.Process(target=ignore) for _ in range(4)]
    for p in stubborn: p.start()
    time.sleep(.5); t0 = time.time()
    assert quit_cleanly(run_exit=lambda: None, grace=.5) == 8
    assert time.time() - t0 < 4, f'{time.time()-t0:.1f}s: one shared grace, not one per child'
    assert not [p for p in pool if alive(p)], 'pool workers outlived quit_cleanly'
    assert signal.getsignal(signal.SIGTERM) == signal.SIG_DFL, 'a second SIGTERM must kill a hung quit'
    mid = fork.Process(target=nested, args=(sys.argv[1],)); mid.start()
    while not os.path.exists(sys.argv[1]) or not open(sys.argv[1]).read(): time.sleep(.1)
    grand = [int(x) for x in open(sys.argv[1]).read().split()]
    os.kill(mid.pid, signal.SIGKILL); mid.join()
    end = time.time() + 5
    while [g for g in grand if alive(g)] and time.time() < end: time.sleep(.2)
    assert not [g for g in grand if alive(g)], 'workers outlived a parent that was SIGKILLed'
    print('ok')
'''
if sys.platform == 'darwin':
    with tempfile.TemporaryDirectory() as d:
        r = subprocess.run([sys.executable, '-c', _fork_probe, f'{d}/grand'], capture_output=True, text=True, timeout=120)
    assert r.stdout.strip() == 'ok', r.stdout + r.stderr

In [ ]:
#| export
NO_SUBSTITUTION = ('NSAutomaticQuoteSubstitutionEnabled', 'NSAutomaticDashSubstitutionEnabled',
                   'NSAutomaticTextReplacementEnabled', 'NSAutomaticSpellingCorrectionEnabled',
                   'NSAutomaticPeriodSubstitutionEnabled', 'NSAutomaticCapitalizationEnabled',
                   'NSAutomaticInlinePredictionEnabled')

In [ ]:
#| export
def quiet_text_substitution():
    "Turn off the system's autocorrect for this app. True when it took."
    if sys.platform != 'darwin': return False
    try:
        from Foundation import NSUserDefaults
        d = NSUserDefaults.standardUserDefaults()
        for k in NO_SUBSTITUTION: d.setBool_forKey_(False, k)
        return True
    except Exception as e:
        print(f'  the system may still autocorrect what you type: {errstr(e)}')
        return False

macOS corrects text as you type: it turns straight quotes into curly ones, a pair of hyphens into a dash, and fixes spelling. In a code editor that corrupts what you type. `quiet_text_substitution` turns these corrections off for this app only, by setting the keys in `NO_SUBSTITUTION` in the app's user defaults, and returns whether that worked. On other platforms it does nothing and returns `False`.

In [ ]:
#| export
def keep_running_in_dock(on_folder=None, recent=None):
    "Hide the last window rather than quit, and fill the Dock menu. Reopen shows one window. Cmd+Q quits."
    global _on_folder, _on_recent
    try:
        from webview.platforms.cocoa import BrowserView
        _on_folder, _on_recent = on_folder, recent
        BrowserView.WindowDelegate, BrowserView.AppDelegate = dock_delegates()
        return True
    except Exception as e:
        print(f'  the last window will quit {APP_NAME}: {errstr(e)}')
        return False

`keep_running_in_dock` installs the hide-on-close behaviour and the Dock menu described above. Call it before the first window is created, because pywebview uses the delegate classes it has at that moment. `on_folder` is called with a path chosen from the Dock menu, and `recent` returns the host's recent folders. It returns `True` when the delegates were installed, or prints why and returns `False` if pywebview's Cocoa platform cannot be loaded.

In [ ]:
#| hide
if sys.platform != 'darwin': test_eq(quiet_text_substitution(), False)
if find_spec('webview') is None:
    test_eq(keep_running_in_dock(print), False)
    test_is(_on_folder, None)                  

In [ ]:
#| export
WINDOW_KW = dict(min_size=MIN_SIZE, background_color='#171b20', text_select=True, zoomable=True,
                 easy_drag=False)

`WINDOW_KW` holds the settings every window is created with: the minimum size, which is `MIN_SIZE`, the dark background shown while a page loads, text selection, zoom, and no dragging of the window by its content. The title and the size are left out because they are the only things that differ from one window to the next. A window opened at launch and one opened an hour later for another folder therefore behave the same.

In [ ]:
#| exec_doc
WINDOW_KW

In [ ]:
#| hide
test_is(WINDOW_KW['min_size'], MIN_SIZE)
assert not {'title', 'width', 'height'} & set(WINDOW_KW), 'the per-window arguments stay out of it'

In [ ]:
#| export
_api = None

In [ ]:
#| export
#: The open windows that have a key, by key, so `open_window` raises one rather than opening it twice.
keyed_windows = {}
_windows = keyed_windows

In [ ]:
#| export
def shell_api():
    "The one `ShellApi` every window shares."
    global _api
    if _api is None: _api = ShellApi()
    return _api

Every window shares one `ShellApi`, which `shell_api` creates the first time it is called. `run_shell` sets the host's callbacks on it before opening any window, so windows opened later get the same callbacks.

In [ ]:
#| exec_doc
shell_api() is shell_api()

In [ ]:
#| export
def system_dark():
    "Whether macOS is in dark mode. False elsewhere."
    if sys.platform != 'darwin': return False
    try:
        from Foundation import NSUserDefaults
        return NSUserDefaults.standardUserDefaults().stringForKey_('AppleInterfaceStyle') == 'Dark'
    except Exception: return False

def window_background(background=None):
    "The colour a window opens on: `background`, a `(light, dark)` pair by the system's mode, else the last one set, else the default."
    if isinstance(background, (tuple, list)): background = background[system_dark()]
    return next(c for c in (background, shell_api().background, WINDOW_KW['background_color']) if is_colour(c))

`window_background` decides the colour a window opens on. A host passes one colour, or a `(light, dark)` pair for a theme that follows the system, and `system_dark` picks from the pair. With nothing given, or with something that is not `#rrggbb`, a window takes the colour a page last set through `ShellApi.set_background`, and failing that the dark default in `WINDOW_KW`.

In [ ]:
#| hide
_b = shell_api().background
try:
    shell_api().background = None
    test_eq(window_background(), WINDOW_KW['background_color'])
    test_eq(window_background('#efe6d6'), '#efe6d6')
    test_eq(window_background(('#efe6d6', '#221d18')), ('#221d18' if system_dark() else '#efe6d6'))
    test_eq(window_background('nonsense'), WINDOW_KW['background_color'])
    shell_api().set_background('#252a33')
    test_eq(window_background(), '#252a33')
    if sys.platform != 'darwin': test_eq(system_dark(), False)
    class _W: background_color, uid = '#000000', 'none'
    w = _W(); paint_background(w, '#252a33'); test_eq(w.background_color, '#252a33')
finally: shell_api().background = _b

In [ ]:
#| export
def make_window(title=None, size=None, background=None):
    "A window on the splash, waiting for a URL, on `window_background(background)`."
    import webview
    w, h = size or window_size()
    bg = window_background(background)
    STORAGE.mkdir(parents=True, exist_ok=True)
    window = webview.create_window(title or APP_NAME, html=splash(title or APP_NAME, background=bg), width=w, height=h,
                                   js_api=shell_api(), **{**WINDOW_KW, 'background_color': bg})
    if sys.platform == 'darwin': window.events.loaded += arm_drops
    window.titlebar = shell_api().titlebar()
    if window.titlebar != 'native':
        # Only the drag region itself moves the window, so a press on a button in it stays a click.
        webview.settings['DRAG_REGION_DIRECT_TARGET_ONLY'] = True
        window.events.before_show += lambda: unified_titlebar(window, bg)
    elif not shell_api().title_text and sys.platform == 'darwin': window.events.before_show += lambda: hide_title_text(window)
    return window


`make_window` creates a window showing the splash page, titled `title` (the app's name by default) and sized `size` (from `window_size` by default). It does not load a URL; `open_window` and `run_shell` do that once the server answers. It also creates the `STORAGE` folder if needed. The window and its splash take `window_background(background)`, so with no `background` a window opens on the colour the page last set. On macOS each page the window loads gets `arm_drops`, so a file dropped on it opens.


## A unified title bar

A host whose page draws its own top row can merge that row with the window's title bar. The host sets `shell_api().unified = True` before it opens windows, and each platform then does what is native to it:

- On macOS the page lies under a transparent title bar with no title. An empty unified toolbar makes the bar 52 points high, so the traffic lights sit centred in the page's top row. The page leaves about 80 pixels free at its left for them. The title is still set, so the Window menu, Mission Control and tools that read the title still have it.
- On Windows the native caption stays, with its minimise, maximise and close buttons, snap layouts and keyboard behaviour. `tint_caption` paints it in the page's background, with dark mode on a dark one, so the caption and the page's top row read as one bar. Windows 11 takes the caption colours; Windows 10 takes only dark mode.
- Elsewhere nothing changes.

The page learns the mode two ways: `with_titlebar` puts it on the first URL as `titlebar=<mode>`, so the page can lay out its first paint, and `ShellApi.titlebar` answers it once pywebview's bridge is up. The page marks the row that drags the window with pywebview's `pywebview-drag-region` class. WebKit ignores `-webkit-app-region`, so pywebview's own drag code moves the window, and only a press on the marked element itself starts a drag.

In [ ]:
#| export
TITLEBARS = {'darwin': 'mac-unified', 'win32': 'win-caption'}
#: AppKit's `NSWindowStyleMaskFullSizeContentView`, `NSWindowTitleHidden` and `NSWindowToolbarStyleUnified`.
FULL_SIZE_CONTENT, TITLE_HIDDEN, TOOLBAR_UNIFIED = 1 << 15, 1, 3
#: DWM attributes: immersive dark mode, caption colour and caption text colour.
DWMWA_DARK, DWMWA_CAPTION, DWMWA_TEXT = 20, 35, 36

def titlebar_mode(platform=None):
    "How a unified window shows its title bar on `platform`: `mac-unified`, `win-caption`, or `native` where nothing changes."
    return TITLEBARS.get(platform or sys.platform, 'native')

def with_titlebar(url, mode):
    "`url` with `titlebar=<mode>` in its query, unless `mode` is `native`."
    if mode == 'native': return url
    from urllib.parse import urlsplit, urlunsplit, parse_qsl, urlencode
    u = urlsplit(url)
    q = [(k, v) for k, v in parse_qsl(u.query, keep_blank_values=True) if k != 'titlebar'] + [('titlebar', mode)]
    return urlunsplit(u._replace(query=urlencode(q)))

In [ ]:
#| exec_doc
titlebar_mode('darwin'), titlebar_mode('win32'), titlebar_mode('linux'), with_titlebar('http://k.localhost:5001/?a=1#x', 'mac-unified')

In [ ]:
#| hide
test_eq(with_titlebar('http://127.0.0.1:5001/', 'native'), 'http://127.0.0.1:5001/')
test_eq(with_titlebar('http://127.0.0.1:5001/?titlebar=win-caption', 'mac-unified'), 'http://127.0.0.1:5001/?titlebar=mac-unified')
test_eq(shell_api().titlebar(), 'native')

In [ ]:
#| export
def off_main_thread(fn, name='kavacha-shell'):
    "Run `fn(arg)` on a thread of its own; `create_window` builds nothing on the main thread."
    def go(arg): threading.Thread(target=fn, args=(arg,), daemon=True, name=name).start()
    return go

Native events, such as a click in the Dock menu or a folder dropped on the app's icon, arrive on the main thread, inside the GUI loop, and pywebview cannot build a window from there. `off_main_thread` wraps `fn` so that each call runs on a new daemon thread named `name` instead. The example records the name of the thread its callback ran on.

In [ ]:
#| exec_doc
seen, done = [], Event()
go = off_main_thread(lambda p: (seen.append((p, threading.current_thread().name)), done.set()))
go('/proj/demo'); done.wait(5)
seen

In [ ]:
#| export
def bring_forward():
    "Activate the app, and drop the hidden-window entries whose window is on screen again. macOS only."
    if sys.platform != 'darwin': return
    try:
        from AppKit import NSApp
        from PyObjCTools.AppHelper import callAfter
    except ImportError: return
    def go():
        NSApp.activateIgnoringOtherApps_(True)
        hidden_windows[:] = [w for w in hidden_windows if not w.isVisible()]
    callAfter(go)

def open_window(url, title=None, key=None, url_wait=90):
    "A window on `url`, raised rather than opened twice when `key` already has one."
    title = title or APP_NAME
    bring_forward()
    if key and (window := keyed_windows.get(key)) is not None:
        try: window.show()
        except Exception as e: print(f'  could not raise the window for {title}: {errstr(e)}')
        return window
    window = make_window(title)
    if key: keyed_windows[key] = window
    _point(window, url, url_wait)
    return window

In [ ]:
#| export
def _point(window, url, url_wait=90):
    "Wait for the server, then show the workspace. A window that never answers says so."
    if wait_for_http(url, timeout=url_wait): window.load_url(with_titlebar(url, getattr(window, 'titlebar', 'native')))
    else: window.load_html(_unreachable(url, getattr(window, 'background_color', None) or WINDOW_KW['background_color']))

`open_window` opens a window on `url` while the app is running, for example when the user opens another folder. `key` identifies the window, such as the folder's path. If a window with that key is already open, `open_window` shows it, even if it was hidden to the Dock, instead of opening a second one. Either way it activates the app with `bring_forward`, so a window opened from Finder is not left behind other apps. The windows that have a key are kept in the dictionary `keyed_windows`, which `run_shell` also fills. Otherwise it creates a window, shows the splash, waits up to `url_wait` seconds for the server and loads the page, or shows the unreachable page if the server never answers. It blocks while it waits, so call it from a thread made by `off_main_thread`.

The example uses stand-in windows. The first is brought to the front. The second fails to come to the front, so `open_window` prints why and returns it as it is.

In [ ]:
#| hide
#| exec_doc
class _Win:
    "Stands in for the pywebview window: `open_window` and `_point` use three of its methods."
    def __init__(self, fail=False): self.calls, self.fail = [], fail
    def show(self):
        if self.fail: raise RuntimeError('the window is gone')
        self.calls.append('show')
    def load_url(self, u): self.calls.append(('url', u))
    def load_html(self, h): self.calls.append(('html', h))

In [ ]:
#| exec_doc
keyed_windows['demo'], keyed_windows['gone'] = _Win(), _Win(fail=True)
u = 'http://127.0.0.1:8000/'
open_window(u, 'Demo', key='demo').calls, open_window(u, 'Notes', key='gone').calls

In [ ]:
#| hide
w = _Win(); _point(w, url)
test_eq(w.calls, [('url', url)])
w = _Win(); _point(w, dead, url_wait=.3)
test_eq(w.calls[0][0], 'html')
assert dead in w.calls[0][1], 'the window says which address did not answer'
keyed_windows.clear()

In [ ]:
#| export
def unify_nswindow(nsw, toolbar=None):
    "Lay `nsw`'s content under a transparent title bar with no title, 52 points high so the traffic lights centre in the page's top row."
    nsw.setStyleMask_(nsw.styleMask() | FULL_SIZE_CONTENT)
    nsw.setTitlebarAppearsTransparent_(True)
    nsw.setTitleVisibility_(TITLE_HIDDEN)
    if toolbar is None:
        from AppKit import NSToolbar
        toolbar = NSToolbar.alloc().initWithIdentifier_('kavacha-unified')
    toolbar.setShowsBaselineSeparator_(False)
    nsw.setToolbar_(toolbar)
    if nsw.respondsToSelector_('setToolbarStyle:'): nsw.setToolbarStyle_(TOOLBAR_UNIFIED)
    return 'mac-unified'

def colorref(colour):
    "`#rrggbb` as a Windows COLORREF, `0x00bbggrr`."
    r, g, b = (int(colour[i:i+2], 16) for i in (1, 3, 5))
    return r | g << 8 | b << 16

def _dwm_set(hwnd, attr, value):
    import ctypes
    v = ctypes.c_int(value)
    return ctypes.windll.dwmapi.DwmSetWindowAttribute(ctypes.c_void_p(hwnd), attr, ctypes.byref(v), ctypes.sizeof(v)) == 0

def tint_caption(hwnd, background, set_attr=None):
    "Paint the Windows caption of `hwnd` in `background`, with dark mode and light text on a dark one. Whether dark mode took."
    if not is_colour(background): return False
    set_attr = set_attr or _dwm_set
    dark = not is_light(background)
    ok = set_attr(hwnd, DWMWA_DARK, int(dark))
    set_attr(hwnd, DWMWA_CAPTION, colorref(background))
    set_attr(hwnd, DWMWA_TEXT, colorref('#e6edf3' if dark else '#1f2328'))
    return bool(ok)

def _native(window, platform):
    "pywebview's own view of `window` on `platform`, or None before it is built."
    if platform == 'darwin': from webview.platforms.cocoa import BrowserView
    elif platform == 'win32': from webview.platforms.winforms import BrowserView
    else: return None
    return BrowserView.instances.get(window.uid)

def hide_title_text(window, native=None):
    "Draw no title text in macOS `window`'s native title bar; the title stays set. Whether it took. Call on the GUI thread."
    try:
        if (v := native or _native(window, 'darwin')) is None: return False
        v.window.setTitleVisibility_(TITLE_HIDDEN); return True
    except Exception as e:
        print(f'  hidden title: {errstr(e)}'); return False

def unified_titlebar(window, background=None, platform=None, native=None):
    "Merge pywebview `window`'s title bar with the page's top row as this platform does it natively; the mode it gave. Call on the GUI thread."
    platform = platform or sys.platform
    if (mode := titlebar_mode(platform)) == 'native': return mode
    try:
        if (v := native or _native(window, platform)) is None: return 'native'
        if mode == 'mac-unified': return unify_nswindow(v.window)
        window.hwnd = int(v.Handle.ToInt64())
        tint_caption(window.hwnd, background or window_background())
        return mode
    except Exception as e:
        print(f'  unified title bar: {errstr(e)}')
        return 'native'

`unified_titlebar` is what `make_window` runs before a unified window is shown. It finds pywebview's own window object, the `NSWindow` on macOS or the WinForms form on Windows, and calls `unify_nswindow` or `tint_caption` on it. A Windows window keeps its handle as `window.hwnd`, so `paint_background` tints the caption again each time the page changes its theme. A failure prints why and leaves the window as it was.

The examples use stand-in windows, so no real window opens. The Mac stand-in records the calls AppKit would get; the Windows one records the DWM attributes.

In [ ]:
#| exec_doc
class _NSWindow:
    "Stands in for an NSWindow: it records what `unify_nswindow` sets."
    def __init__(self): self.mask, self.set = 15, {}
    def styleMask(self): return self.mask
    def setStyleMask_(self, m): self.mask = m
    def setTitlebarAppearsTransparent_(self, v): self.set['transparent'] = v
    def setTitleVisibility_(self, v): self.set['title'] = v
    def setToolbar_(self, t): self.set['toolbar'] = t
    def respondsToSelector_(self, s): return s == 'setToolbarStyle:'
    def setToolbarStyle_(self, v): self.set['style'] = v
class _Toolbar:
    def setShowsBaselineSeparator_(self, v): self.separator = v
class _PyWin: uid = 'w1'
nsw = _NSWindow()
unify_nswindow(nsw, _Toolbar()), nsw.mask, nsw.set['title']

In [ ]:
#| hide
test_eq(nsw.mask & FULL_SIZE_CONTENT, FULL_SIZE_CONTENT)
test_eq(nsw.mask & 15, 15)   # titled, closable, miniaturizable, resizable: the traffic lights stay
test_eq((nsw.set['transparent'], nsw.set['title'], nsw.set['style']), (True, TITLE_HIDDEN, TOOLBAR_UNIFIED))
test_eq(nsw.set['toolbar'].separator, False)
_old = _NSWindow(); _old.respondsToSelector_ = lambda s: False
unify_nswindow(_old, _Toolbar()); assert 'style' not in _old.set, 'macOS before 11 has no toolbar style and keeps its default'
class _Mac:
    "A pywebview Cocoa view whose NSWindow is a stand-in; AppKit is never reached for the toolbar."
    def __init__(self): self.window = _NSWindow()
with patch.dict(globals(), unify_nswindow=lambda w, toolbar=None: (w.set.update(done=True), 'mac-unified')[1]):
    m = _Mac(); test_eq(unified_titlebar(_PyWin(), platform='darwin', native=m), 'mac-unified'); test_eq(m.window.set['done'], True)
test_eq(unified_titlebar(_PyWin(), platform='linux', native=_Mac()), 'native')
m = _Mac(); test_eq(hide_title_text(_PyWin(), native=m), True)
test_eq(m.window.set['title'], TITLE_HIDDEN); assert 'transparent' not in m.window.set and 'toolbar' not in m.window.set, 'only the text goes; the bar stays native'

In [ ]:
#| exec_doc
calls = []
tint_caption(0x1234, '#161b22', lambda h, a, v: calls.append((h, a, hex(v))) or True), calls

In [ ]:
#| hide
test_eq(colorref('#0969da'), 0xda6909)
test_eq(calls, [(0x1234, DWMWA_DARK, '0x1'), (0x1234, DWMWA_CAPTION, hex(colorref('#161b22'))), (0x1234, DWMWA_TEXT, hex(colorref('#e6edf3')))])
calls = []; tint_caption(7, '#f6f8fa', lambda h, a, v: calls.append((a, v)) or True)
test_eq(calls[0], (DWMWA_DARK, 0)); test_eq(calls[2], (DWMWA_TEXT, colorref('#1f2328')))
test_eq(tint_caption(7, 'nonsense', lambda *a: True), False)
class _Handle:
    def ToInt64(self): return 0x5150
class _Form: Handle = _Handle()
seen = []
with patch.dict(globals(), _dwm_set=lambda h, a, v: seen.append((h, a)) or True):
    w = _PyWin(); test_eq(unified_titlebar(w, '#0d1117', platform='win32', native=_Form()), 'win-caption')
    test_eq(w.hwnd, 0x5150); test_eq([a for _, a in seen], [DWMWA_DARK, DWMWA_CAPTION, DWMWA_TEXT])
    seen.clear(); w.background_color = None; paint_background(w, '#ffffff')
    test_eq(seen[0], (0x5150, DWMWA_DARK))   # a theme change tints the caption again
class _Broken:
    @property
    def Handle(self): raise RuntimeError('no handle yet')
test_eq(unified_titlebar(_PyWin(), platform='win32', native=_Broken()), 'native')
shell_api().unified = True
try:
    test_eq(shell_api().titlebar(), titlebar_mode())
    w = _Win(); _point(w, url)
    test_eq(w.calls, [('url', url)])   # a window made before the host opted in keeps its url
    w = _Win(); w.titlebar = 'mac-unified'; _point(w, url)
    test_eq(w.calls, [('url', with_titlebar(url, 'mac-unified'))])
finally: shell_api().unified = False

In [ ]:
#| export
def run_shell(urls, titles=(), url_wait=90, gui=None, icon=None, size=None, on_ready=None,
              on_open_folder=None, keys=(), bar=(), lookup=None, on_action=None, on_recent=None, background=None,
              on_open_file=None):
    "Open one native window per workspace URL and block until they close."
    ok, why = shell_ready()
    if not ok: raise RuntimeError(why)
    import webview
    urls = list(urls)
    if not urls: raise ValueError('the desktop shell needs at least one workspace URL')
    titles = list(titles) + [''] * (len(urls) - len(titles))
    keys = list(keys) + [None] * (len(urls) - len(keys))
    open_folder = off_main_thread(on_open_folder, 'kavacha-open-folder') if on_open_folder else None
    open_file = off_main_thread(on_open_file, 'kavacha-open-file') if on_open_file else None
    shell_api().on_open_folder, shell_api().on_open_file = open_folder, open_file
    if background is not None: shell_api().set_background(window_background(background))
    if sys.platform == 'darwin':
        quiet_text_substitution()
        keep_running_in_dock(open_folder, on_recent)
        quit_on_sigterm()
    windows = [make_window(t, size) for t in titles]
    for key, window in zip(keys, windows):
        if key: keyed_windows[key] = window
    if open_folder or open_file: watch_open_events(open_folder, open_file)
    def load(*_):
        "Once the GUI loop is up: wait for the server, then point each window at it."
        for window, url in zip(windows, urls): _point(window, url, url_wait)
        if on_ready is None: return
        try: on_ready()
        except Exception as e: print(f'  desktop shell: {errstr(e)}')
    built = menus(bar, lookup, on_action) if bar else []
    webview.start(load, gui=gui or why, debug=bool(os.environ.get(f'{ENV_PREFIX}WEBVIEW_DEBUG')),
                  private_mode=False, storage_path=str(STORAGE), icon=icon, menu=built)

`run_shell` is the entry point. It opens one window per URL in the list `urls`, titled with the matching entry of `titles`, and runs the GUI loop until the app quits. Call it on the main thread, after starting the servers on another thread or in another process. Each window shows the splash until its server answers, for up to `url_wait` seconds. It raises a `RuntimeError` if no window can be opened here and a `ValueError` if `urls` is empty.

The other arguments are optional:

- `size` is a `(width, height)` tuple for every window; without it `window_size` decides.
- `gui` overrides the pywebview GUI that `shell_ready` picks, and `icon` is the window icon.
- `on_ready` is called once every window has loaded its page.
- `on_open_folder` and `on_recent` are the host's callbacks for opening a folder and for listing recent ones. They serve `ShellApi`, the Dock menu and folders dropped on the app's icon.
- `on_open_file` is called with the path of each file the user opens with the app from Finder, by double-click or Open With, whether the app is running or being launched by it.
- `keys` gives each window a key, so that `open_window` can find it later.
- `background` is the colour every window opens on, as `window_background` reads it: a host with themes passes its page background, so the window does not flash another colour before the page paints.
- `bar`, `lookup` and `on_action` describe the menu bar, as explained below.

On macOS it also turns off text substitution, keeps the app in the Dock when its last window closes, and makes SIGTERM quit cleanly. Setting the environment variable `<prefix>WEBVIEW_DEBUG` turns on pywebview's debug mode.

In [ ]:
#| hide
if find_spec('webview') is None:
    test_fail(lambda: run_shell(['http://127.0.0.1:8000/']), contains='pywebview is not installed',
              exc=RuntimeError)

In [ ]:
#| export
def menus(bar, lookup, on_action, run=None):
    "Build the macOS menu bar, or return an empty list."
    if sys.platform != 'darwin': return []
    try:
        import webview
        webview.settings['SHOW_DEFAULT_MENUS'] = False
        built, specs = menu_bar(bar, lookup, on_action, run)
        install_menu_patch(specs, lookup)
        return built
    except Exception as e:
        print(f'  no menu bar: {errstr(e)}')
        return []

`menus` builds the macOS menu bar from the host's table `bar` with `menu_bar` and `install_menu_patch`, both below, and turns off pywebview's default menus. `lookup` and `on_action` are explained with `menu_chord` and `menu_bar`, and `run` is the function `Js` rows use, `eval_js` by default. On other platforms, or if anything fails, it returns an empty list and the window has no menu bar.

In [ ]:
#| hide
if sys.platform != 'darwin': test_eq(menus([], {}, lambda *_: None), [])

A file or folder opened with the app reaches it in up to two ways: the delegate's `application:openFiles:`, and the raw `odoc` handler below. `open_paths` is where both end. It calls `on_folder` with each folder in `paths` and `on_file` with each file, and returns `False` if some path had no callback for its kind, which is how the delegate tells Finder the open failed. A path is delivered once within `OPEN_TTL` seconds, so a single open that both routes see does not open two windows.

In [ ]:
#| export
OPEN_TTL = 2.
_seen = {}

def open_paths(paths, on_folder=None, on_file=None):
    "Deliver each path in `paths` once to `on_folder` or `on_file`. False if one had no callback."
    now = time.monotonic()
    for p in [p for p, t in _seen.items() if now - t > OPEN_TTL]: del _seen[p]
    ok = True
    for p in map(str, paths):
        cb = on_folder if os.path.isdir(p) else on_file
        if cb is None: ok = False
        elif p not in _seen:
            _seen[p] = now
            cb(p)
    return ok

In [ ]:
#| hide
_d = Path(tempfile.mkdtemp()); _f = _d/'x.md'; _f.write_text('hi')
_got = []
def _rec(kind): return lambda p: _got.append((kind, p))
_seen.clear()
test_eq(open_paths([_d, _f], _rec('dir'), _rec('file')), True)
test_eq(_got, [('dir', str(_d)), ('file', str(_f))])
test_eq(open_paths([_f, _d], _rec('dir'), _rec('file')), True)
test_eq(len(_got), 2)
_seen.clear()
test_eq(open_paths([_d, _f], None, _rec('file')), False)
test_eq(_got[2:], [('file', str(_f))])
test_eq(open_paths([_f], _rec('dir'), None), False)
test_eq(len(_got), 3)

In [ ]:
#| export
DROP_JS = r"""(() => {
  if (window.__kavachaDrop) return; window.__kavachaDrop = true;
  const files = e => [...(e.dataTransfer?.types || [])].includes('Files'), ours = new WeakSet();
  addEventListener('dragover', e => { if (files(e) && !e.defaultPrevented) { e.preventDefault(); e.dataTransfer.dropEffect = 'copy'; } });
  addEventListener('drop', e => {
    if (!files(e)) return;
    const names = [...e.dataTransfer.files].map(f => f.name);
    setTimeout(() => window.pywebview?.api?.dropped?.(names, ours.has(e)), 0);
  }, true);
  addEventListener('drop', e => { if (files(e) && !e.defaultPrevented) { e.preventDefault(); ours.add(e); } });
})();"""

def take_dropped(names):
    "The full paths pywebview kept for the dropped files `names`, in order. It forgets the rest, so a later drop never gets a stale path."
    from unicodedata import normalize
    from urllib.parse import unquote
    try: from webview.dom import _dnd_state
    except ImportError: return []
    nfc = lambda s: normalize('NFC', unquote(str(s)))
    held, out = list(_dnd_state['paths']), []
    for n in map(nfc, names or ()):
        if (hit := next((h for h in held if nfc(h[0]) == n), None)) is None: continue
        held.remove(hit)
        out.append(unquote(hit[1]))
    _dnd_state['paths'].clear()
    return out

def arm_drops(window):
    "Make a file dropped on `window` from outside reach `ShellApi.dropped`, unless the page takes it first."
    from webview.dom import _dnd_state
    # pywebview's macOS view keeps a drop's full paths only while some drop listener is registered.
    _dnd_state['num_listeners'] = max(1, _dnd_state['num_listeners'])
    _dnd_state['paths'].clear()
    window.run_js(DROP_JS)


A file or folder dragged from Finder onto the window, rather than onto the app's icon, is a drop in the web page. The page learns only the file's name, never its location. pywebview's macOS view reads the full paths from the drag and keeps them, but only while some drop listener is registered, and `arm_drops` makes sure one is counted. `make_window` calls `arm_drops` on every page load, and it adds `DROP_JS` to the page.

`DROP_JS` lets a file drag land anywhere, and it leaves every drop the page handles itself to the page: a page handler that calls `preventDefault` or `stopPropagation` keeps the drop, as an editor that takes a file's text does. When the drop is over, the script calls `dropped` with the files' names and whether the drop was left to the window. `take_dropped` exchanges the names for the paths pywebview kept, and forgets the rest. `dropped` then opens the paths through `open_paths`, with the same callbacks as a Finder open. A page script that calls `dropped` gets nothing that was not dropped from outside, because pywebview keeps no path for a drag that starts in the page, and each page load forgets the paths of a drop that never reached `dropped`.

In [ ]:
_dnd = types.SimpleNamespace(_dnd_state={'num_listeners': 0, 'paths': []})
_api4, _got4 = ShellApi(), []
_api4.on_open_folder, _api4.on_open_file = _rec('dir'), _rec('file')
_got.clear(); _seen.clear()
with patch.dict(sys.modules, {'webview.dom': _dnd}):
    _dnd._dnd_state['paths'] += [('x.md', str(_f)), (_d.name, str(_d)), ('old.png', '/gone/old.png')]
    test_eq(_api4.dropped(['x.md', _d.name]), [str(_f), str(_d)])
    test_eq(_got, [('file', str(_f)), ('dir', str(_d))])
    test_eq(_dnd._dnd_state['paths'], [])
    _dnd._dnd_state['paths'].append(('x.md', str(_f))); _seen.clear()
    test_eq(_api4.dropped(['x.md'], False), [])
    test_eq((len(_got), _dnd._dnd_state['paths']), (2, []))
    test_eq(_api4.dropped(['x.md']), [])
    _dnd._dnd_state['paths'] += [('x.md', '/a/x.md'), ('x.md', '/b/x.md'), ('cafe\u0301.md', '/c/caf.md')]
    test_eq(take_dropped(['x.md', 'x.md', 'caf\u00e9.md']), ['/a/x.md', '/b/x.md', '/c/caf.md'])
    _dnd._dnd_state['paths'].append(('x.md', str(_f)))
    w = types.SimpleNamespace(js=[], run_js=lambda s: w.js.append(s))
    arm_drops(w)
    test_eq((_dnd._dnd_state['num_listeners'], _dnd._dnd_state['paths'], w.js), (1, [], [DROP_JS]))


In [ ]:
#| hide
if sys.platform == 'darwin' and find_spec('webview') is not None:
    import AppKit
    class _Sender:
        def __init__(self): self.replies = []
        def replyToOpenOrPrint_(self, r): self.replies.append(r)
    _app, _s, _api = dock_delegates()[1].alloc().init(), _Sender(), shell_api()
    _old = _api.on_open_folder, _api.on_open_file
    _api.on_open_folder, _api.on_open_file = _rec('dir'), _rec('file')
    _got.clear(); _seen.clear()
    _app.application_openFiles_(_s, [str(_d), str(_f)])
    test_eq(_got, [('dir', str(_d)), ('file', str(_f))])
    test_eq(_s.replies, [AppKit.NSApplicationDelegateReplySuccess])
    test_eq(_app.application_openFile_(_s, str(_f)), True)
    test_eq(len(_got), 2)
    _api.on_open_file = None; _seen.clear()
    _app.application_openFiles_(_s, [str(_f)])
    test_eq(_s.replies[-1], AppKit.NSApplicationDelegateReplyFailure)
    _api.on_open_folder, _api.on_open_file = _old

In [ ]:
#| export
def watch_open_events(on_folder=None, on_file=None):
    "Answer the `odoc` Apple Event Finder sends a running app. macOS only, best-effort."
    if sys.platform != 'darwin': return None
    try: from Foundation import NSAppleEventManager, NSURL
    except ImportError: return None
    def code(s): return int.from_bytes(s.encode(), 'big')
    def handle(event, _reply):
        try:
            items = event.paramDescriptorForKeyword_(code('----'))
            paths = []
            for i in range(1, (items.numberOfItems() or 0) + 1):
                url = items.descriptorAtIndex_(i).stringValue()
                if not url: continue
                p = NSURL.URLWithString_(url).path() if url.startswith('file:') else url
                if p: paths.append(str(p))
            open_paths(paths, on_folder, on_file)
        except Exception as e: print(f'  open event: {errstr(e)}')
    try:
        mgr = NSAppleEventManager.sharedAppleEventManager()
        mgr.setEventHandler_andSelector_forEventClass_andEventID_(
            OpenHandler.alloc().init_with(handle), b'handleEvent:reply:',
            code('aevt'), code('odoc'))
        return handle
    except Exception as e:
        print(f'  could not register the open handler: {errstr(e)}')
        return None

When the user drops a folder on the app's Dock icon, or opens a folder with the app from Finder, macOS sends the running app an "open documents" Apple Event, `odoc`. `watch_open_events` registers a handler for it that hands the paths in the event to `open_paths`, which calls `on_folder` for each folder and `on_file` for each file. `run_shell` registers it with the host's callbacks, already wrapped by `off_main_thread`. The app delegate receives the same opens through `application:openFiles:`, and `open_paths` keeps a path that arrives both ways from being delivered twice. It returns the handler, or `None` on other platforms or if registration fails.

In [ ]:
#| hide
if sys.platform != 'darwin': test_is(watch_open_events(print), None)

In [ ]:
#| export
try:                                     # pragma: no cover - macOS only
    import objc
    from Foundation import NSObject
    class RecentTarget(NSObject):
        "A target for a Dock menu row."
        def init_with(self, fn):
            self = objc.super(RecentTarget, self).init()
            if self is None: return None
            self._fn = fn
            return self
        def openRecent_(self, sender):
            path = sender.representedObject()
            if self._fn and path: self._fn(str(path))

    class OpenHandler(NSObject):
        "An Apple Event handler must be a selector on an Objective-C object."
        def init_with(self, fn):
            self = objc.super(OpenHandler, self).init()
            if self is None: return None
            self._fn = fn
            return self
        def handleEvent_reply_(self, event, reply): self._fn(event, reply)
except Exception:                        # pragma: no cover - everywhere else
    OpenHandler = RecentTarget = None
_OpenHandler, _RecentTarget = OpenHandler, RecentTarget
_all_ = ['OpenHandler', 'RecentTarget']

AppKit can only call methods on Objective-C objects, so the Dock menu rows and the Apple Event handler each need a small `NSObject` subclass that passes the call on to a Python function: `RecentTarget` for a Dock menu row and `OpenHandler` for the event. Where pyobjc is not installed, both classes are `None`, and the functions that use them only run on macOS.

In [ ]:
#| hide
if find_spec('objc') is None: test_is(RecentTarget, None); test_is(OpenHandler, None)
# the old private names are the same objects, so state shared through either stays one state
test_is(_RecentTarget, RecentTarget); test_is(_OpenHandler, OpenHandler); test_is(_dock_delegates, dock_delegates)
test_is(_hidden, hidden_windows); test_is(_windows, keyed_windows)

In [ ]:
#| export
CMD, CTRL, ALT, SHIFT = 1 << 20, 1 << 18, 1 << 19, 1 << 17

In [ ]:
#| export
_MODS = {'mod': CMD, 'cmd': CMD, 'ctrl': CTRL, 'alt': ALT, 'opt': ALT, 'shift': SHIFT,
         'hyper': CMD | CTRL | ALT | SHIFT}

In [ ]:
#| export
_NAMED = {'up': '', 'down': '', 'left': '', 'right': '',
          'pageup': '', 'pagedown': '', 'home': '', 'end': '',
          'delete': '', 'enter': '\r', 'return': '\r', 'tab': '\t', 'space': ' ',
          'backspace': '\x08', 'escape': '\x1b'}

A menu bar shows keyboard shortcuts next to its items, and pressing one runs the item even while the web page has focus. kavacha writes a shortcut as a chord, such as `mod+s`: modifier names joined by `+`, then a key, where `mod` means Cmd on a Mac. AppKit wants the same shortcut as a key equivalent, which is a single character, plus a modifier mask, a number with one bit per modifier key. `CMD`, `CTRL`, `ALT` and `SHIFT` are those bits, defined here so that chords can be parsed and tested on any platform.

In [ ]:
#| export
_US_SHIFT = dict(zip("`1234567890-=[]\\;',./", '~!@#$%^&*()_+{}|:"<>?'))

def mac_key(chord):
    "`(keyEquivalent, modifierMask)` for one chord, or None when AppKit cannot spell it."
    parts = str(chord or '').split()[0].split('+') if chord else []
    if not parts or not parts[-1]: return None
    mask, shift, base = 0, False, parts[-1]
    for p in parts[:-1]:
        if (m := _MODS.get(p.lower())) is None or p.lower() == 'hyper': return None
        if m == SHIFT: shift = True
        else: mask |= m
    shift |= len(base) == 1 and base.isupper()
    if (named := _NAMED.get(base.lower())) is not None: return named, mask | (SHIFT if shift else 0)
    if len(base) != 1: return None
    b = base.lower()
    if not shift: return b, mask
    if b.isalpha(): return b.upper(), mask
    return (_US_SHIFT[b], mask) if b in _US_SHIFT else (b, mask | SHIFT)

`mac_key` converts one chord to a `(key, mask)` pair. It accepts the modifiers `mod`, `cmd`, `ctrl`, `alt`, `opt` and `shift`, an upper-case letter as shorthand for shift and that letter, and named keys such as `up`, `enter`, `tab` and `escape`. For a sequence of chords, such as `mod+k mod+s`, it uses only the first.

AppKit matches a key equivalent against the character the key types, so Shift is written into the key and not into the mask: `mod+shift+o` is `O` with Command, and `mod+shift+/` is `?` with Command, the character Shift gives on a US keyboard. A named key such as an arrow types no character, so it keeps the Shift bit. `mac_key` returns `None` for `hyper`, which a window manager takes, and for a chord AppKit cannot show as a key equivalent, such as one with a function key or an unknown modifier like `meta`.

In [ ]:
#| exec_doc
mac_key('mod+p'), mac_key('mod+f5'), mac_key('meta+p')

In [ ]:
#| exec_doc
test_eq(mac_key('mod+P'), mac_key('mod+shift+p'))
test_eq(mac_key('mod+shift+o'), ('O', CMD))
test_eq(mac_key('mod+shift+/'), ('?', CMD))
test_eq(mac_key('mod+k mod+s'), mac_key('mod+k'))

In [ ]:
#| hide
test_eq(mac_key('mod+tab'), ('\t', CMD))
test_eq(mac_key('escape'), ('\x1b', 0))
test_eq(len(mac_key('mod+up')[0]), 1)
test_eq(mac_key('mod+shift+up'), (_NAMED['up'], CMD | SHIFT))
test_eq(mac_key('alt+shift+1'), ('!', ALT))
test_eq(mac_key('alt+i'), ('i', ALT))
test_eq(mac_key('mod+shift+enter'), ('\r', CMD | SHIFT))
for bad in (None, '', 'mod+', 'mod+f5', 'meta+p', 'mod+ctrl+f12', 'hyper+x', 'hyper+k'): test_is(mac_key(bad), None)

In [ ]:
#| export
def menu_chord(action, lookup):
    "The AppKit chord for `action`: its first chord with a modifier, not reserved, from a global binding or the only scope with that chord."
    k = lookup(action)
    if k is None or not k.bound or (k.scope != 'global' and getattr(k, 'shared', True)): return None
    reserved = {mac_key(c) for c in RESERVED}
    for c in k.keys:
        if not (set(c.split('+')[:-1]) & {'mod', 'cmd', 'ctrl', 'alt', 'opt'}): continue
        if (got := mac_key(c)) is not None and got not in reserved: return got

A menu shortcut takes the key press before the web page sees it, so only some of the host's bindings should become one. The host's keymap is reached through `lookup`, a function that takes an action name and returns a `Binding` from `kavacha.menus`, or `None`. `menu_chord` looks up `action` and returns the AppKit chord of the binding's first chord that uses a modifier other than shift and is not in `RESERVED`, and only when the binding either applies everywhere (its scope is `global`) or is the only binding of that chord in any scope (`shared` is false). A bare key such as `/`, or an editor chord that another pane uses too, stays with the page. A lookup that returns some other object with no `shared` attribute gets chords for its global bindings only.

In [ ]:
#| export
def _title(row, lookup):
    "The label for an action row: the keymap's, or the action with its underscores opened up."
    k = lookup(row.action)
    t = (k.label if k and k.label else row.action.replace('_', ' '))
    return t[:1].upper() + t[1:]

In [ ]:
#| exec_doc
KEYS = {'save':         Binding(('mod+s',), 'Save'),
        'quick_open':   Binding(('mod+shift+o',)),
        'find_in_file': Binding(('mod+f',), 'Find', scope='editor', shared=True),
        'cell_delete':  Binding(('mod+alt+backspace',), scope='editor'),
        'comment':      Binding(('/',), 'Comment'),
        'quit':         Binding(('mod+q',)),
        'orphan':       Binding()}
{a: menu_chord(a, KEYS.get) for a in KEYS}

`save` and `quick_open` are bound, global and use a modifier, so both get their chord, and `quick_open` has Shift written into its key. `find_in_file` applies only in the editor and another scope binds its chord, so it stays with the page. `cell_delete` is scoped too, but no other scope binds its chord, so it gets one. `comment` has no modifier, `quit` is reserved and `orphan` has no key, so each of them gets `None`, as does an action the keymap does not know.

In [ ]:
#| hide
test_eq(menu_chord('save', KEYS.get), ('s', CMD))
test_eq(menu_chord('quick_open', KEYS.get), ('O', CMD))
test_is(menu_chord('find_in_file', KEYS.get), None)
test_eq(menu_chord('cell_delete', KEYS.get), ('\x08', CMD | ALT))
test_is(menu_chord('comment', KEYS.get), None)
test_is(menu_chord('quit', KEYS.get), None)
test_is(menu_chord('orphan', KEYS.get), None)
test_is(menu_chord('never_heard_of_it', KEYS.get), None)
test_is(menu_chord('x', {'x': Binding(('hyper+n',))}.get), None)
test_is(menu_chord('x', {'x': Binding(('ctrl+alt+h',))}.get), None)
test_eq(menu_chord('x', {'x': Binding(('hyper+n', 'n', 'alt+w'))}.get), ('w', ALT))

A row's label is the keymap's `label`. When there is none, `_title` uses the action name with its underscores replaced by spaces and the first letter capitalised, so a row is never blank and never shows a raw identifier.

In [ ]:
#| exec_doc
_title(MenuItem('save'), KEYS.get), _title(MenuItem('quick_open'), KEYS.get)

In [ ]:
#| hide
test_eq(_title(MenuItem('save'), KEYS.get), 'Save')       
test_eq(_title(MenuItem('quick_open'), KEYS.get), 'Quick open')
test_eq(_title(MenuItem('never_seen'), KEYS.get), 'Never seen')

In [ ]:
#| export
def menu_bar(bar, lookup, on_action, run=None):
    "Build pywebview menus from `bar`, and the row table `_decorate` uses to add native shortcuts and selectors."
    from webview.menu import Menu, MenuAction, MenuSeparator
    run = run or _menu_run
    menus, specs = [], {}
    for title, rows in bar:
        items = []
        for row in rows:
            if isinstance(row, MenuItem) and not row.action: items.append(MenuSeparator()); continue
            if isinstance(row, Std):
                specs[(title, row.title)] = row
                items.append(MenuAction(row.title, lambda: None))
            elif isinstance(row, Js):
                items.append(MenuAction(row.title, (lambda e: lambda: run(e))(row.expr)))
            else:
                name = _title(row, lookup)
                specs[(title, name)] = row
                items.append(MenuAction(name, (lambda a: lambda: on_action(a))(row.action)))
        menus.append(Menu(title, items))
    return menus, specs

`menu_bar` turns the host's menu table into pywebview menus. `bar` is a list of `(title, rows)` pairs, one per menu, and each row is one of the classes in `kavacha.menus`. A `MenuItem` calls `on_action` with its action name, and a `MenuItem` without an action is a separator. A `Std` row is handled by AppKit, and a `Js` row runs a JavaScript expression in the focused window through `run`, which is `eval_js` by default. `eval_js` returns what the expression evaluated to, or `None`.

pywebview cannot set a shortcut or an AppKit selector on a menu item, so `menu_bar` also returns a table that maps `(menu title, row title)` to the row. `_decorate`, below, uses it to add those natively.

In [ ]:
#| export
def eval_js(js):
    "Run one expression in the window that has focus, and return what it evaluated to."
    import webview
    if (window := webview.active_window()) is None: return None
    try: return window.evaluate_js(js)
    except Exception as e:
        print(f'  menu: {errstr(e)}')
        return None

_menu_run = eval_js

In [ ]:
#| export
_menu_patched = False

In [ ]:
#| export
def install_menu_patch(specs, lookup):
    "Reapply AppKit decoration whenever pywebview rebuilds the menu."
    global _menu_patched
    if _menu_patched: return False
    try:
        from webview.platforms.cocoa import BrowserView
    except Exception as e:
        print(f'  no menu bar: {errstr(e)}')
        return False
    original = BrowserView._recreate_menus
    def recreate(self, user_menu):
        main = original(self, user_menu)
        try:
            _decorate(main, specs, lookup)
            main = exact_bar(main)
        except Exception as e: print(f'  menu bar: {errstr(e)}')
        return main
    BrowserView._recreate_menus = recreate
    _menu_patched = True
    return True


pywebview rebuilds the native menu bar at times of its own choosing, and each rebuild would lose those additions. `install_menu_patch` wraps pywebview's menu-building method so that `_decorate` runs after every rebuild, and `exact_bar` then moves the rows into a bar that matches chords exactly. It installs the wrapper once per process and returns whether it did so on this call.


In [ ]:
#| hide
if find_spec('webview') is None:
    test_eq(install_menu_patch({}, KEYS.get), False)
    test_eq(_menu_patched, False)

In [ ]:
#| export
def _decorate(main, specs, lookup):
    "Chords onto the rows that may have one, and the responder chain onto the rows AppKit owns."
    import AppKit
    for i in range(main.numberOfItems()):
        sub = main.itemAtIndex_(i).submenu()
        if sub is None: continue
        title = str(sub.title())
        for j in range(sub.numberOfItems()):
            item = sub.itemAtIndex_(j)
            spec = specs.get((title, str(item.title())))
            if spec is None: continue
            if isinstance(spec, Std):
                item.setTarget_(None)
                item.setAction_(spec.selector)
                got = mac_key(f'{spec.mods}+{spec.key}') if spec.key else None
            else: got = menu_chord(spec.action, lookup)
            if got: item.setKeyEquivalent_(got[0]); item.setKeyEquivalentModifierMask_(got[1])
        if title == 'Window': AppKit.NSApp().setWindowsMenu_(sub)

In [ ]:
#| export
_ExactBar = None

def exact_bar(main):
    "A menu bar holding the rows of `main`, where a row fires only for its own chord."
    global _ExactBar
    import AppKit, objc
    mods = AppKit.NSEventModifierFlagCommand | AppKit.NSEventModifierFlagOption | AppKit.NSEventModifierFlagControl | AppKit.NSEventModifierFlagShift
    if _ExactBar is None:
        class KavachaExactBar(AppKit.NSMenu):
            def performKeyEquivalent_(self, e):
                m, c = e.modifierFlags() & mods, e.charactersIgnoringModifiers()
                if m & AppKit.NSEventModifierFlagShift and c and c.lower() == c.upper():
                    for top in self.itemArray():
                        if (sub := top.submenu()) is None: continue
                        sub.update()
                        for j, i in enumerate(sub.itemArray()):
                            if i.isEnabled() and i.keyEquivalent() == c and i.keyEquivalentModifierMask() & mods == m:
                                sub.performActionForItemAtIndex_(j)
                                return True
                return objc.super(KavachaExactBar, self).performKeyEquivalent_(e)
        _ExactBar = KavachaExactBar
    bar = _ExactBar.alloc().initWithTitle_(main.title())
    for item in list(main.itemArray()):
        main.removeItem_(item)
        bar.addItem_(item)
    return bar


AppKit lets a row with no Shift take the shifted chord of a key that has no case: a row on ⌥↩ also runs for ⌥⇧↩, before the row that has ⌥⇧↩ gets it. `exact_bar` moves the rows into a bar that, for a Shift chord on such a key, first runs the row whose chord is exactly the one pressed. Other chords go to AppKit as before. The test builds both rows on a bar and presses each chord.

In [ ]:
#| hide
if sys.platform == 'darwin' and find_spec('AppKit') is not None:
    import AppKit
    from Foundation import NSObject
    AppKit.NSApplication.sharedApplication()
    _ran = []
    class _Rows(NSObject):
        def row_(self, sender): _ran.append(str(sender.title()))
    _t, _plain, _sub = _Rows.alloc().init(), AppKit.NSMenu.alloc().init(), AppKit.NSMenu.alloc().initWithTitle_('Run')
    _top = AppKit.NSMenuItem.alloc().init(); _top.setSubmenu_(_sub); _plain.addItem_(_top)
    for _n, _m in (('insert', ALT), ('selection', ALT | SHIFT)):
        _i = AppKit.NSMenuItem.alloc().initWithTitle_action_keyEquivalent_(_n, 'row:', '\r')
        _i.setKeyEquivalentModifierMask_(_m); _i.setTarget_(_t); _sub.addItem_(_i)
    def _press(bar, mask):
        e = AppKit.NSEvent.keyEventWithType_location_modifierFlags_timestamp_windowNumber_context_characters_charactersIgnoringModifiers_isARepeat_keyCode_(
            AppKit.NSEventTypeKeyDown, (0, 0), mask, 0, 0, None, '\r', '\r', False, 36)
        _ran.clear(); bar.performKeyEquivalent_(e); return list(_ran)
    test_eq(_press(_plain, ALT | SHIFT), ['insert'])
    _bar = exact_bar(_plain)
    test_eq((_plain.numberOfItems(), _bar.numberOfItems()), (0, 1))
    test_eq((_press(_bar, ALT | SHIFT), _press(_bar, ALT)), (['selection'], ['insert']))


`_decorate` walks the native menu bar and finishes each row it has a spec for. A `Std` row loses its target and gets its AppKit selector, so the action goes to whatever has focus, along with the row's own chord. An action row gets the chord `menu_chord` allows, if any. The menu titled `Window` becomes the app's Windows menu, where macOS lists the open windows.

### The menu bar as data

A launcher that builds its own native menu, in another language, needs the same table with every chord already resolved. `menu_json` gives it: one `{'title', 'items'}` per menu, where an item is a separator, an action row, a `Std` row with its selector, or a `Js` row with its expression.

On a Mac, a row carries `key` and `mask` when it has a chord. The chords come from `menu_chord` and `mac_key`, the same as `_decorate` uses.

Off a Mac (`mac=False`, for example a GTK launcher on Linux), a row carries `chord`, the label to show, such as `Ctrl+Shift+P`. A row has `accel`, a GTK accelerator, only when the launcher may bind it natively. Those rows are the `PC_NATIVE` launcher rows (Quit, Close Window and Full Screen) and the actions in `native`, such as a new window, which the page never needs. Every other chord is a label only, so the key press still reaches the page: Ctrl+C in a terminal is still an interrupt. `pc_key` gives the `Std` chords, and `pc_menu_chord` picks an action's chord as `menu_chord` does, but it refuses the chords in `pc_reserved`. A row that `pc_key` removes is left out, and so is a separator with nothing to separate. `mac=None` uses the platform the code runs on.

In [ ]:
#| export
def _bar_row(row, lookup):
    if isinstance(row, MenuItem) and not row.action: return {'sep': True}
    if isinstance(row, Js): return {'title': row.title, 'js': row.expr}
    if isinstance(row, Std): out, k = {'title': row.title, 'selector': row.selector}, mac_key(f'{row.mods}+{row.key}') if row.key else None
    else: out, k = {'title': _title(row, lookup), 'action': row.action}, menu_chord(row.action, lookup)
    return out | (dict(key=k[0], mask=k[1]) if k else {})

def pc_menu_chord(action, lookup):
    "The chord off a Mac for `action`: its first chord with a modifier that the desktop does not keep, from a global binding or the only scope with that chord."
    k = lookup(action)
    if k is None or not k.bound or (k.scope != 'global' and getattr(k, 'shared', True)): return None
    return next((pc_chord(c) for c in k.keys if _chord(c)[0] & {'mod', 'ctrl', 'alt'} and not pc_reserved(c)), None)

def _pc_row(row, lookup, native):
    if isinstance(row, MenuItem) and not row.action: return {'sep': True}
    if isinstance(row, Js): return {'title': row.title, 'js': row.expr}
    if isinstance(row, Std):
        if (c := pc_key(row)) is None: return None
        out, safe = {'title': row.title, 'selector': row.selector}, row.selector in PC_NATIVE
    else: out, c, safe = {'title': _title(row, lookup), 'action': row.action}, pc_menu_chord(row.action, lookup), row.action in native
    if not c: return out
    return out | {'chord': chord_glyph(c, mac=False)} | ({'accel': gtk_accel(c)} if safe else {})

def _tidy(items):
    "`items` without the removed rows, and without a separator at either end or next to another."
    out = []
    for i in items:
        if i is not None and not (i.get('sep') and (not out or out[-1].get('sep'))): out.append(i)
    return out[:-1] if out and out[-1].get('sep') else out

def menu_json(bar, lookup, mac=None, native=()):
    "`bar` as JSON-ready menus for a native launcher: AppKit `key` and `mask` on a Mac, else a `chord` label and a GTK `accel` on the rows safe to bind."
    if (sys.platform == 'darwin' if mac is None else mac): return [{'title': t, 'items': [_bar_row(r, lookup) for r in rows]} for t, rows in bar]
    return [{'title': t, 'items': _tidy([_pc_row(r, lookup, set(native)) for r in rows])} for t, rows in bar]

In [ ]:
#| export
def launcher_json(lookup=lambda a: None, mac=None):
    "`LAUNCHER_ROWS` as menu rows: the chords a native launcher owns (Quit, Hide, Close Window, Undo, ...)."
    from kavacha.menus import LAUNCHER_ROWS
    return menu_json([('', LAUNCHER_ROWS)], lookup, mac=mac)[0]['items']

In [ ]:
test_eq(launcher_json(mac=True)[3], {'title': 'Close Window', 'selector': 'performClose:', 'key': 'W', 'mask': CMD})
test_eq(launcher_json(mac=True)[-1]['mask'], CMD | CTRL)

In [ ]:
#| exec_doc
menu_json([('File', [MenuItem('save'), MenuItem(), Std('Close Window', 'performClose:', 'w', 'mod+shift'), Js('Reload', 'location.reload()')])], KEYS.get, mac=True)

In [ ]:
#| hide
got = menu_json([('File', [MenuItem('save'), MenuItem(), Std('Close Window', 'performClose:', 'w', 'mod+shift'), Js('Reload', 'location.reload()'), MenuItem('comment')])], KEYS.get, mac=True)
test_eq(got, [{'title': 'File', 'items': [{'title': 'Save', 'action': 'save', 'key': 's', 'mask': CMD}, {'sep': True},
    {'title': 'Close Window', 'selector': 'performClose:', 'key': 'W', 'mask': CMD}, {'title': 'Reload', 'js': 'location.reload()'},
    {'title': 'Comment', 'action': 'comment'}]}])

In [ ]:
#| hide
pc = dict(KEYS, new_window=Binding(('mod+shift+n',), 'New Window'), cell_above=Binding(('mod+alt+up', 'a'), scope='editor'),
          copy_path=Binding(('mod+alt+l', 'mod+alt+c')))
bar = [('File', [MenuItem('new_window'), MenuItem('save'), MenuItem(), Std('Close Window', 'performClose:', 'w', 'mod+shift'),
                 Std('Quit', 'terminate:', 'q')]),
       ('Edit', [Std('Copy', 'copy:', 'c'), MenuItem('cell_above'), MenuItem('copy_path')]),
       ('View', [Std('Enter Full Screen', 'toggleFullScreen:', 'f', 'mod+ctrl')]),
       ('Window', [Std('Minimize', 'performMiniaturize:', 'm'), Std('Zoom', 'performZoom:'), MenuItem(), MenuItem('quick_open'), MenuItem(),
                   Std('Bring All to Front', 'arrangeInFront:')])]
test_eq(menu_json(bar, pc.get, mac=False, native=('new_window',)), [
    {'title': 'File', 'items': [{'title': 'New Window', 'action': 'new_window', 'chord': 'Ctrl+Shift+N', 'accel': '<Control><Shift>n'},
        {'title': 'Save', 'action': 'save', 'chord': 'Ctrl+S'}, {'sep': True},
        {'title': 'Close Window', 'selector': 'performClose:', 'chord': 'Ctrl+Shift+W', 'accel': '<Control><Shift>w'},
        {'title': 'Quit', 'selector': 'terminate:', 'chord': 'Ctrl+Q', 'accel': '<Control>q'}]},
    {'title': 'Edit', 'items': [{'title': 'Copy', 'selector': 'copy:', 'chord': 'Ctrl+C'}, {'title': 'Cell above', 'action': 'cell_above'},
        {'title': 'Copy path', 'action': 'copy_path', 'chord': 'Ctrl+Alt+C'}]},
    {'title': 'View', 'items': [{'title': 'Enter Full Screen', 'selector': 'toggleFullScreen:', 'chord': 'F11', 'accel': 'F11'}]},
    {'title': 'Window', 'items': [{'title': 'Quick open', 'action': 'quick_open', 'chord': 'Ctrl+Shift+O'}]}])
test_eq(menu_json(bar, pc.get, mac=True)[2]['items'][0]['mask'], CMD | CTRL)

In [ ]:
#| hide
srv.shutdown()